# `aidp-alh` live test — API Key + inline OCI config (catalog-sync side)

**Live-test row 3.** Refreshes the AIDP external catalog metadata from ALH using inline-PEM OCI auth, then reads the synced table via Spark.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import from_inline_pem
import oci

config = from_inline_pem(
    user_ocid=os.environ['OCI_USER_OCID'],
    tenancy_ocid=os.environ['OCI_TENANCY_OCID'],
    fingerprint=os.environ['OCI_FINGERPRINT'],
    private_key_pem=os.environ['OCI_PRIVATE_KEY_PEM'],
    region=os.environ['OCI_REGION'],
)
# A control-plane sanity check — proves the config works without writing a PEM file.
identity = oci.identity.IdentityClient(config=config)
print('user:', identity.get_user(config['user']).data.name)


In [ ]:
# Downstream Spark read against the externally-cataloged ALH table.
df = spark.read.table(os.environ['ALH_EXTERNAL_CATALOG_TABLE'])
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-alh',
    'auth': 'apikey-catalog-sync',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
